In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import os
import matplotlib.pyplot as plt


print(f"Pytorch version: {torch.__version__}")
print(f"Cuda Availablle: {torch.cuda.is_available()}")

if torch.cuda.is_available():
  print(f"CUDA version:{torch.version.cuda}")
  print(f"Gpu device: {torch.cuda.get_device_name(0)}")

Pytorch version: 2.10.0+cpu
Cuda Availablle: False


In [ ]:
sample_dir = 'samples'

if not os.path.exists(sample_dir):
  os.makedirs(sample_dir)
  print(f"Created Directory:{sample_dir}")

Created Directory:samples


In [ ]:
##HYperParameters

latent_size=64
hidden_size=256
image_size=784
num_epochs=100
batch_size=100


print("="*100)
print("GAN HyperParameters COnfiguration")
print("="*70)
print(f"LatentSpaceDim : {latent_size}")
print(f"HiddenLayerSize : {hidden_size}")
print(f"ImageSize(flattened) : {image_size}")
print(f"TrainingEpochs: {num_epochs}")
print(f"Batch SIze : {batch_size}")
print("="*70)

GAN HyperParameters COnfiguration
LatentSpaceDim : 64
HiddenLayerSize : 256
ImageSize(flattened) : 784
TrainingEpochs: 100
Batch SIze : 100


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nCompute Device: {device}")

if device.type=='cuda':
  print(f"Gpu Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Compute Device: cpu


In [ ]:
def load_mnist_data(batch_size):
    transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(
          mean=[0.5],
          std=[0.5]
      )
    ])

    ##Load MNIST Training dataset with transformations
    mnist = torchvision.datasets.FashionMNIST(
      root='./data/',
      train=True,
      transform=transform,
      download=True
    )

    ##Create DataLoader for efficient batchprocessing
    data_loader = torch.utils.data.DataLoader(
      dataset=mnist,
      batch_size=batch_size,
      shuffle=True
    )
    print(f"\nDataset loaded successfully!!")
    print(f"Total training images: {len(mnist)}")
    print(f"Batches per epoch: {len(data_loader)}")
    print(f"Image shape: (1, 28, 28)")

    return data_loader

In [ ]:
##Initialize data loader with configured batch size
data_loader = load_mnist_data(batch_size)

100%|██████████| 26.4M/26.4M [00:01<00:00, 20.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.64MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.6MB/s]


Dataset loaded successfully!!
Total training images: 60000
Batches per epoch: 600
Image shape: (1, 28, 28)


#**Model Architecture of our GAN**

In [ ]:
class Generator(nn.Module):
  def __init__(self, latent_size, hidden_size, image_size):
    super(Generator, self).__init__()

    self.model = nn.Sequential(

        ##FIrst transformation: Latent Space -> hidden representation

        nn.Linear(latent_size, hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size, hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size, image_size),
        nn.Tanh()
    )
  def forward(self, z):

    return self.model(z)


generator = Generator(latent_size, hidden_size, image_size).to(device)

print("\n" + "="*70)
print("GENERATOR ARCHITECTURE")
print("="*70)
print(generator)
print("="*70)
print(f"Total Parameters: {sum(p.numel() for p in generator.parameters()):,}")
print("="*70)






GENERATOR ARCHITECTURE
Generator(
  (model): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=784, bias=True)
    (5): Tanh()
  )
)
Total Parameters: 283,920


In [ ]:
test_noise = torch.randn(1, latent_size).to(device)
test_output = generator(test_noise)
print(f"\nGenerator Test:")
print(f"Input shape: {test_noise.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Output range: [{test_output.min().item():.3f}, {test_output.max().item():.3f}]")


Generator Test:
Input shape: torch.Size([1, 64])
Output shape: torch.Size([1, 784])
Output range: [-0.314, 0.342]


In [ ]:
class Discriminator(nn.Module):

  def __init__(self, image_size, hidden_size):
    super(Discriminator, self).__init__()

    self.model = nn.Sequential(
        nn.Linear(image_size, hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size, hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size, 1),
        nn.Sigmoid()
    )

  def forward(self, x):

    return self.model(x)


discriminator = Discriminator(image_size, hidden_size).to(device)

print("\n" + "="*70)
print("DISCRIMINATOR ARCHITECTURE")
print("="*70)
print(discriminator)
print("="*70)
print(f"Total Parameters: {sum(p.numel() for p in discriminator.parameters()):,}")
print("\n" + "="*70)




DISCRIMINATOR ARCHITECTURE
Discriminator(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=1, bias=True)
    (5): Sigmoid()
  )
)
Total Parameters: 267,009



In [ ]:
test_image = torch.randn(1, image_size).to(device)
test_classification = discriminator(test_image)

print(f"\nDiscriminator Test:")
print(f"Input shape:{ test_image.shape}")
print(f"Output shape:{ test_classification.shape}")
print(f"Output value: {test_classification.item():.4f} (probability of being real)")



Discriminator Test:
Input shape:torch.Size([1, 784])
Output shape:torch.Size([1, 1])
Output value: 0.4906 (probability of being real)


# **Training Configuration**

In [ ]:
criterion = nn.BCELoss()

d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0002)
g_optimizer = torch.optim.Adam(generator.parameters(), lr=0.0002)


print("\nOptimization Configuration:")
print(f"Loss function: Binary Cross-Entropy")
print(f"Optimizer: Adam")


Optimization Configuration:
Loss function: Binary Cross-Entropy
Optimizer: Adam


**Auxiliary Functions for Training**

In [ ]:
def denorm(x):

  out = (x+1) / 2
  return out.clamp(0, 1)
def reset_grad():
  d_optimizer.zero_grad()
  g_optimizer.zero_grad()

# **TRAINING LOOP**

In [ ]:
total_step = len(data_loader)
d_losses = []
g_losses = []

print("\n"+ "="*70)
print("Starting GAN Training")
print("="*70)
print(f"Total Epochs {num_epochs}")
print(f"Stepts per EPoch: {total_step}")
print(f"Total Training Steps: {num_epochs * total_step}")
print("="*70 + "\n")

for epoch in range(num_epochs):
  epoch_d_loss = 0
  epoch_g_loss = 0

  for i, (images, _) in enumerate(data_loader):
    ##PREPARE DATA

    images = images.reshape(batch_size, -1).to(device)

    real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)

    ###TRAIN DISCRIMINATOR
    ##Objective : Maximize log(D(x)) + log(1-D(G(z)))
    #Equivalently: Minimize BCE loss on real and fake images

    ##Compute loss on real images
    outputs = discriminator(images)
    d_loss_real = criterion(outputs, real_labels)
    real_score = outputs

    ##Compute loss on fake images
    #generate fake images from random noise
    z = torch.randn(batch_size, latent_size).to(device)
    fake_images = generator(z)

    ##D shoud output low probability close to 0 for fake images

    outputs = discriminator(fake_images.detach())

    d_loss_fake = criterion(outputs, fake_labels)
    fake_score = outputs

    ##combining oss and update discriminator

    d_loss = d_loss_real + d_loss_fake
    reset_grad()
    d_loss.backward()
    d_optimizer.step()


    ##TRAIN GENERATOR
    ##Objective: Maximize log(D(G(z)))
    #Equivalently: Minimize BCE loss with real labels
    ##We want D to classify fake as real image

    ##Generating fake images from random noise
    z = torch.randn(batch_size, latent_size).to(device)
    fake_images = generator(z)

    outputs = discriminator(fake_images)
    g_loss = criterion(outputs, real_labels)

    reset_grad()
    g_loss.backward()
    g_optimizer.step()

    ##LOGGING AND MONITIORING

    epoch_d_loss += d_loss.item()
    epoch_g_loss += g_loss.item()

    if (i+1) % 200 == 0:
      print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_step}],'
            f'D_loss: {d_loss.item():.4f}, G_loss: {g_loss.item():.4f},'
            f'D(x): {real_score.mean().item():.2f}, D(G(z)): {fake_score.mean().item():.2f}')

  d_losses.append(epoch_d_loss / total_step)
  g_losses.append(epoch_g_loss / total_step)



if (epoch + 1) % 1 == 0:
  fake_images = fake_images.reshape(fake_images.size(0), 1, 28, 28)
  torchvision.utils.save_image(
      denorm(fake_images),
      os.path.join(sample_dir, f'fake_images-{epoch+1}.png')
  )
  print("\n" + "="*70)
  print("TRAINING COMPLETED")
  print("="*70)
  print(f"FInal D loss:{d_losses[-1]:.4f}")
  print(f"FInal G loss:{g_losses[-1]:.4f}")
  print("="*70)



Starting GAN Training
Total Epochs 100
Stepts per EPoch: 600
Total Training Steps: 60000

Epoch [1/100], Step [200/600],D_loss: 0.0592, G_loss: 3.5826,D(x): 0.99, D(G(z)): 0.05
Epoch [1/100], Step [400/600],D_loss: 0.0911, G_loss: 4.3489,D(x): 0.99, D(G(z)): 0.08
Epoch [1/100], Step [600/600],D_loss: 0.1431, G_loss: 3.4220,D(x): 0.94, D(G(z)): 0.07
Epoch [2/100], Step [200/600],D_loss: 0.0526, G_loss: 4.1790,D(x): 0.98, D(G(z)): 0.03
Epoch [2/100], Step [400/600],D_loss: 0.1390, G_loss: 3.4582,D(x): 0.96, D(G(z)): 0.05
Epoch [2/100], Step [600/600],D_loss: 0.0671, G_loss: 6.2877,D(x): 0.98, D(G(z)): 0.01
Epoch [3/100], Step [200/600],D_loss: 0.1319, G_loss: 4.9134,D(x): 0.94, D(G(z)): 0.04
Epoch [3/100], Step [400/600],D_loss: 0.3901, G_loss: 3.7489,D(x): 0.89, D(G(z)): 0.07
Epoch [3/100], Step [600/600],D_loss: 0.1106, G_loss: 3.9832,D(x): 0.95, D(G(z)): 0.03
Epoch [4/100], Step [200/600],D_loss: 0.1930, G_loss: 5.2156,D(x): 0.94, D(G(z)): 0.02
Epoch [4/100], Step [400/600],D_loss: 0

In [ ]:
print(f"\nTraining Statistics:")
print(f"EPochs completed: {len(d_losses)}")
print(f"Average D loss: {sum(d_losses)/len(d_losses):.4f}")
print(f"Average G loss: {sum(g_losses)/len(g_losses):.4f}")

In [ ]:
##Visualizing Training Loss Curves


plt.figure(figsize=(12,6))
plt.plot(d_losses, label='Discriminator Loss', linewidth=2, color='#e74c3c',alpha=0.8)
plt.plot(g_losses, label='Generator Loss',linewidth=2, color='#3498db',alpha=0.8)
plt.title('GAN Training Loss Curves: Generator vs Discriminator', fontsize=14
          ,fontweight='bold',pad=20)
plt.xlabel('Epochs', font_size=12, fontweigth='bold')
plt.ylabel('Loss',font_size=12, fontweight='bold')
plt.legend(loc='upper right', fontsize=11, frameon=True, shadow=True)
plt.grid(True, alpha=0.3, linestyle='--')

plt.axhline(y=sum(d_losses)/len(d_losses), color='#e74c3c', linestyle=':',
            alpha=0.5, label=f'Avg D Loss: {sum(d_losses)/len(d_losses):.4f}')
plt.axhline(y=sum(g_losses)/len(g_losses), color='#3498db', linestyle=':',
            alpha=0.5, label=f'Avg G Loss: {sum(g_losses)/len(g_losses):.4f}')

plt.tight_layout()
plt.show()


print("\n" + "="*70)
print("LOSS ANALYSIS")
print("="*70)
print(f"Final D Loss: {d_losses[-1]:.4f}")
print(f"Final G Loss: {g_losses[-1]:.4f}")
print(f"Average D loss: {sum(d_losses)/len(d_losses):.4f}")
print(f"Average G loss: {sum(g_losses)/len(g_losses):.4f}")
print(f"Loss Ratio (G/D): {g_losses[-1]/d_losses[-1]:.4f}")
print("="*70)

In [ ]:
# ============================================================================
# SAVE TRAINED MODEL WEIGHTS
# ============================================================================

# Save the generator and discriminator model checkpoints
# This allows loading the trained models later for generating images
torch.save(generator.state_dict(), 'generator.pth')
torch.save(discriminator.state_dict(), 'discriminator.pth')

print("\n" + "=" * 70)
print("MODEL WEIGHTS SAVED")
print("=" * 70)
print("Generator weights saved to: generator.pth")
print("Discriminator weights saved to: discriminator.pth")
print("=" * 70)

In [ ]:
# ============================================================================
# VISUALIZE REAL VS GENERATED IMAGES
# ============================================================================

# Create comprehensive comparison visualization
fig, axes = plt.subplots(2, 10, figsize=(16, 4))

# Display real images from training data
print("\n" + "=" * 70)
print("REAL VS GENERATED IMAGE COMPARISON")
print("=" * 70)

# Get a batch of real images
real_batch = next(iter(data_loader))
real_images = real_batch[0][:10]

# Display real images
for i in range(10):
    axes[0, i].imshow(real_images[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Real Images', fontweight='bold',
                              loc='left', fontsize=11, pad=10)

# Generate and display fake images
generator.eval()  # Set to evaluation mode
with torch.no_grad():
    z = torch.randn(10, latent_size).to(device)
    fake_images = generator(z)
    fake_images = fake_images.view(-1, 1, 28, 28)
    fake_images = denorm(fake_images).cpu()

for i in range(10):
    axes[1, i].imshow(fake_images[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Generated Images', fontweight='bold',
                              loc='left', fontsize=11, pad=10)

plt.suptitle('Real MNIST Images vs GAN-Generated Images',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

generator.train()  # Return to training mode

print("\nVisual Comparison Complete")
print("Top Row:    Real MNIST training images")
print("Bottom Row: GAN-generated synthetic images")
print("=" * 70)